<a href="https://colab.research.google.com/github/roughhawkbit/digi-inno-road-prod/blob/main/analysis/4_0_BERT_DRS_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



Much of the code in this notebook is adapted from Google Gemini responses.

# Setup

In [ ]:
import os
import sys

The below settings appear to be necessary for successful downloading of the pre-trained models from HuggingFace.

In [ ]:
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "15"

from huggingface_hub.utils import _runtime
_runtime._is_google_colab = False
HF_USE_TOKEN = False

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    repo_path = '/content/drive/MyDrive/digi-inno-road-prod'
    if os.path.isdir(repo_path):
      cwd = os.getcwd()
      os.chdir(repo_path)
      !git pull
      os.chdir(cwd)
    else:
      !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git /content/drive/MyDrive/digi-inno-road-prod
      print('Repository cloned into your Google Drive. It is strongly recommended that you copy the credentials.json, sheet.json, and token.json files into the secrets folder before proceeding.')
    sys.path.insert(0, repo_path)
    IN_COLAB = True
except ImportError:
    repo_path = os.path.abspath(os.path.join('../src'))
    IN_COLAB = False

if not repo_path in sys.path:
    sys.path.insert(0, repo_path)

In [ ]:
if IN_COLAB:
  output_path = os.path.join(repo_path, 'analysis', 'outputs')
else:
  output_path = os.path.join('.', 'outputs')
output_path = os.path.abspath(output_path)

# Import packages & data

In [ ]:
!pip install torchao==0.16.0

Conda packages

In [ ]:
import pandas
from peft import get_peft_model, LoraConfig, TaskType
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm
from transformers import BertModel, BertTokenizer

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

Project sourcecode packages

In [ ]:
from innoprod.sheet_tools import get_sheet_dfs
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps

Data

In [ ]:
data = get_sheet_dfs()

INCLUDE_NO_GRANTS_FIRMS = True
if INCLUDE_NO_GRANTS_FIRMS:
  roadmaps_df = pandas.concat([data['Roadmaps'], data['RoadmapsWithoutGrants']])
else:
  roadmaps_df = data['Roadmaps']
roadmaps_df = wrangle_roadmaps(roadmaps_df)

# Model parameters

In [ ]:
# model_name = "bert-large-uncased"
model_name = "bert-base-uncased"

epochs = 50

batch_size = 4
learning_rate = 2e-4

# Maximum number of tokens per input string
max_length = 512

# The fraction of data that will be used for training
training_split = 0.8

random_seed = 10072026

lora_rank = 8
lora_alpha = 16
lora_dropout = 0.1

These are the columns of the dataset that we will use.

In [ ]:
key_questions = [
    'Summary review of Edge Digital diagnostic report & current state and key improvement areas',
    'What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?',
    'Details of any existing Digital Strategy',
    'Level of current Strategic Digital Skills/knowledge in the business',
    'Level of current Technical Digital Skills/knowledge in the business',
    'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples',
    'Summary of the identified problems, including Gap Analysis'
]

drs_col = 'Current Digital Readiness Score (refer to PAS:1040)'

# Transform dataset

Filter the dataset so that only firms where
1.   at least one response has some text; and
2.   the DRS is specified

are included.


In [ ]:
roadmaps_df = roadmaps_df[['Client ID', drs_col] + key_questions]
roadmaps_df[key_questions] = roadmaps_df[key_questions].fillna('')
roadmaps_df[roadmaps_df[key_questions] == 'nan'] = ''
roadmaps_df['Word Count'] = roadmaps_df.apply(lambda row: sum([len(str(row[q]).split()) for q in key_questions]), axis=1)
roadmaps_df = roadmaps_df[roadmaps_df['Word Count'] > 0].drop(columns=['Word Count'])
roadmaps_df = roadmaps_df[roadmaps_df[drs_col].notna()]
roadmaps_df = roadmaps_df.reset_index(drop=True)
# roadmaps_df

In [ ]:
class BusinessReadinessDataset(Dataset):
    """
    Custom Dataset to handle 7 separate text responses per firm
    and map them to a continuous readiness score.
    """
    def __init__(self, dataframe, tokenizer, max_length=max_length):
        self.dataframe = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        # Extract the 7 text responses for this specific firm
        texts = [str(row[col]) for col in key_questions]

        # Tokenize the 7 texts simultaneously
        # batch_encode_plus processes a list of strings and returns stacked tensors
        encodings = self.tokenizer( #.batch_encode_plus
            texts,
            add_special_tokens=True,    # Adds [CLS] and [SEP]
            max_length=self.max_length, # Truncates or pads to this length
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"         # Returns PyTorch tensors
        )

        # encodings['input_ids'] will automatically be shape (7, max_length)
        input_ids = encodings['input_ids']
        attention_mask = encodings['attention_mask']

        # Extract the target score and convert it to a float tensor
        # We use float because we are treating this as a regression task (MSELoss)
        label = torch.tensor(row[drs_col], dtype=torch.float)

        return input_ids, attention_mask, label

In [ ]:
tokenizer = BertTokenizer.from_pretrained(model_name)

full_dataset = BusinessReadinessDataset(roadmaps_df, tokenizer, max_length=max_length)

train_size = int(training_split * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(random_seed)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=True,
    drop_last=False
)

Sanity check the data

In [ ]:
print(f"batch_size: {batch_size}")
print(f"number of questions: {len(key_questions)}")
print(f"max_length: {max_length}")
print("")

for input_ids, attention_mask, labels in train_loader:
    print(f"Input IDs shape:      {input_ids.shape}")       # Expected: [batch_size, len(key_questions), max_length]
    print(f"Attention Mask shape: {attention_mask.shape}")  # Expected: [batch_size, len(key_questions), max_length]
    print(f"Labels shape:         {labels.shape}")          # Expected: [batch_size]
    break # Just testing the first batch

# Document Attention model

In [ ]:
class DocumentAttention(torch.nn.Module):
    """
    Learns to dynamically weight the importance of the 7 different
    qualitative responses for a single firm.
    """
    def __init__(self, hidden_size):
        super().__init__()
        # W matrix for context mapping
        self.attention_weights = torch.nn.Linear(hidden_size, hidden_size)
        # c vector for scoring
        self.context_vector = torch.nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden_states):
        # hidden_states shape: (batch_size, num_questions, hidden_size)

        # 1. Non-linear transformation
        u = torch.tanh(self.attention_weights(hidden_states))

        # 2. Calculate attention scores and apply softmax
        # attention_scores shape: (batch_size, num_questions, 1)
        attention_scores = self.context_vector(u)
        attention_weights = torch.softmax(attention_scores, dim=1)

        # 3. Compute weighted sum of the hidden states
        # aggregated_vector shape: (batch_size, hidden_size)
        aggregated_vector = torch.sum(attention_weights * hidden_states, dim=1)

        return aggregated_vector, attention_weights

# HAN-BERT-LoRA model

In [ ]:
class HANBERTLora(torch.nn.Module):
    """
    End-to-end Hierarchical Attention Network over a LoRA-adapted BERT-large.
    """
    def __init__(self, num_classes=1):
        super().__init__()

        # 1. Load the frozen base BERT model
        base_model = BertModel.from_pretrained(model_name, token=HF_USE_TOKEN)

        # 2. Configure and apply LoRA
        lora_config = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=lora_rank,
            lora_alpha=lora_alpha,
            target_modules=["query", "value"],
            lora_dropout=lora_dropout,
            bias="none"
        )
        # get_peft_model automatically freezes the base model and injects LoRA adapters
        self.bert = get_peft_model(base_model, lora_config)

        hidden_size = base_model.config.hidden_size # 1024 for BERT-large

        # 3. Initialize Firm-Level Attention
        self.document_attention = DocumentAttention(hidden_size)

        # 4. Final Head (Regression for continuous score, or Classification for discrete)
        # Using num_classes=1 implies a regression task for the Digital Readiness Score
        self.classifier = torch.nn.Linear(hidden_size, num_classes)

        self.to(DEVICE)

    def forward(self, input_ids, attention_mask):
        # Expected Input Shapes: (batch_size, num_questions, sequence_length)
        batch_size, num_questions, seq_len = input_ids.size()

        # Flatten the batch and question dimensions to process through BERT
        # New shape: (batch_size * num_questions, sequence_length)
        flat_input_ids = input_ids.view(-1, seq_len)
        flat_attention_mask = attention_mask.view(-1, seq_len)

        # Pass through the LoRA-adapted BERT
        outputs = self.bert(
            input_ids=flat_input_ids,
            attention_mask=flat_attention_mask
        )

        # Extract the [CLS] token embeddings
        # Shape: (batch_size * num_questions, hidden_size)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]

        # Reshape back to group embeddings by firm
        # Shape: (batch_size, num_questions, hidden_size)
        reshaped_embeddings = cls_embeddings.view(batch_size, num_questions, -1)

        # Apply the document-level attention
        aggregated_representation, att_weights = self.document_attention(reshaped_embeddings)

        # Generate the final readiness score prediction
        logits = self.classifier(aggregated_representation)

        # Returning attention weights is highly recommended for academic research
        # as it allows you to explain *which* questions drove the prediction.
        return logits, att_weights

# Accuracy evaluation

In [ ]:
def evaluate_ordinal_metrics(predictions, targets):
    """
    Calculates Exact Accuracy, Adjacent Accuracy, and MAE for an ordinal scale.
    """
    # 1. Round predictions to nearest integer
    rounded_preds = torch.round(predictions)

    # 2. Clamp predictions to the valid ordinal range (1 to 9)
    clamped_preds = torch.clamp(rounded_preds, min=1.0, max=9.0)

    # Ensure targets are float for math operations, but conceptually they are integers
    targets = targets.float()

    # 3. Calculate Exact Match Accuracy
    exact_matches = (clamped_preds == targets).sum().item()
    exact_accuracy = exact_matches / len(targets)

    # 4. Calculate Adjacent (Within-1) Accuracy
    absolute_differences = torch.abs(clamped_preds - targets)
    adjacent_matches = (absolute_differences <= 1).sum().item()
    adjacent_accuracy = adjacent_matches / len(targets)

    # 5. Calculate Mean Absolute Error (MAE)
    mae = absolute_differences.mean().item()

    return exact_accuracy, adjacent_accuracy, mae

# Fine-tune model

In [ ]:
model = HANBERTLora(num_classes=1)

In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=learning_rate,
    weight_decay=0.01  # Standard regularization for transformers
)

In [ ]:
from transformers import get_linear_schedule_with_warmup

In [ ]:
accumulation_steps = 4
total_steps = len(train_loader) * epochs // accumulation_steps

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

In [ ]:
def validate(model, val_loader):
  total_loss = 0
  model.eval()
  all_preds = []
  all_targets = []

  with torch.no_grad():
    i = 1
    for input_ids, attention_mask, labels in val_loader:
      i += 1
      input_ids = input_ids.to(DEVICE)
      attention_mask = attention_mask.to(DEVICE)
      logits, _ = model(input_ids, attention_mask)
      all_preds.append(logits.squeeze())
      all_targets.append(labels)

  all_preds_tensor = torch.cat(all_preds)
  all_targets_tensor = torch.cat(all_targets)
  all_preds_tensor = all_preds_tensor.to(DEVICE)
  all_targets_tensor = all_targets_tensor.to(DEVICE)
  return evaluate_ordinal_metrics(all_preds_tensor, all_targets_tensor)

In [ ]:
# criterion = torch.nn.CrossEntropyLoss()
criterion = torch.nn.MSELoss()

for epoch in range(epochs):
  model.train()
  running_loss = 0.0

  # Progress bar for training tracking
  progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

  optimizer.zero_grad()  # Initialize gradients outside the loop for accumulation

  for i, (input_ids, attention_mask, labels) in enumerate(progress_bar):
    # Move data to GPU
    input_ids = input_ids.to(DEVICE)
    attention_mask = attention_mask.to(DEVICE)
    labels = labels.to(DEVICE).unsqueeze(1)  # Match prediction shape (batch, 1)

    # Forward pass
    predictions, _ = model(input_ids, attention_mask)

    # Calculate loss (scaled by accumulation steps)
    loss = criterion(predictions, labels)
    loss = loss / accumulation_steps
    loss.backward()

    # Update weights only after accumulating enough gradients
    if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
      # Gradient clipping to prevent exploding gradients
      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

      optimizer.step()
      scheduler.step()
      optimizer.zero_grad()

    # Log running loss
    running_loss += loss.item() * accumulation_steps
    progress_bar.set_postfix({'loss': running_loss / (i + 1)})

  # --- Validation Phase ---
  print(f"\n--- Validation for Epoch {epoch+1} ---")
  val_exact_acc, val_adj_acc, val_mae = validate(model, val_loader)
  print(f"Epoch {epoch+1}/{epochs}")
  print(f"  Exact Accuracy:    {val_exact_acc:.4f} ({val_exact_acc*100:.1f}%)")
  print(f"  Adjacent Accuracy: {val_adj_acc:.4f} ({val_adj_acc*100:.1f}%)")
  print(f"  MAE:               {val_mae:.4f}")
  print("-" * 30)